# 02c — Probability, statistics, and uncertainty — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/02.md) · [Course map](../PLAN.md) · [Project](../../projects/stage02/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Compute sample statistics
- Apply conditional probability and Bayes rule
- Estimate uncertainty with reproducible bootstrap samples


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Start here: probability and uncertainty

**Prerequisites:** fractions, percentages, averages, and squares. Probability describes uncertainty under a model; statistics uses observations to estimate or examine that model. Neither removes the need to question how data was collected.

### 1. Outcomes, events, and probability

For a fair six-sided die, the possible outcomes are 1 through 6. An **event** is a collection of outcomes, such as “even,” containing 2, 4, 6. Under equal probabilities, $P(even)=3/6=.5=50\%$. Probabilities lie between 0 and 1. The probability of “not A” is $1-P(A)$.

Do not automatically add probabilities for overlapping events. On a die, “even” and “greater than 3” overlap at 4 and 6. Count each outcome once, or use $P(A\cup B)=P(A)+P(B)-P(A\cap B)$. The union symbol means “A or B”; intersection means “both.”

**Conditional probability** $P(A\mid B)$ reads “probability of A given B.” Restrict attention to outcomes consistent with B. Given an even die result, the probability it exceeds 3 is 2/3 because the remaining outcomes are 2, 4, 6. **Independence** means learning B does not change A's probability. Two separate fair die rolls can be modeled as independent; two properties of the same roll often cannot.

### 2. Bayes using counts

Imagine 1,000 messages: 100 spam and 900 legitimate. A filter flags 90 of the spam and 90 legitimate messages. Among the 180 flagged messages, 90 are spam. Thus $P(spam\mid flagged)=90/180=.5$, even though $P(flagged\mid spam)=.9$.

Bayes' rule expresses the same counting logic: $P(A\mid B)=P(B\mid A)P(A)/P(B)$ when $P(B)>0$. The **prior** $P(A)$ is the rate before observing B; the **posterior** is the updated probability after observing B. Reversing a conditional without considering base rates is a common error.

### 3. Random variables and distributions

A **random variable** assigns a numeric value to an uncertain outcome. A Bernoulli variable is 1 for success and 0 for failure, with success probability $p$. A Binomial($n,p$) variable counts successes in $n$ independent Bernoulli trials with the same probability. Its expected count is $np$; with 20 trials and $p=.3$, expect 6, but individual samples can differ.

A **distribution** tells us how probabilities are allocated. A normal distribution is a symmetric bell-shaped model described by mean $\mu$ and standard deviation $\sigma$. Not every dataset is normal. A random seed makes a pseudo-random sequence repeatable; it does not make the sample representative or the assumed distribution correct.

### 4. Mean, variance, and units

For `[2,4,6]`, the mean is $(2+4+6)/3=4$. Deviations from the mean are `[-2,0,2]`, and squared deviations are `[4,0,4]`. Population variance divides their sum by $n$: $8/3$. Sample variance, used to estimate a population variance from an IID sample with an estimated mean, divides by $n-1$: $8/2=4$.

Why $n-1$? Once the sample mean is fixed, deviations sum to zero, so only $n-1$ deviations are free to vary. The correction makes this variance estimator unbiased under its assumptions. With one observation, sample variance is undefined; population variance of that one-value collection is zero.

Standard deviation is the square root of variance. Here sample standard deviation is 2. If measurements are in metres, variance is in square metres and standard deviation returns to metres. The mean is sensitive to extreme values; also inspect the median, spread, and raw observations.

### 5. Covariance and correlation

Covariance measures whether paired values move together. For $x=[1,2,3]$, $y=[2,4,6]$, their means are 2 and 4. Paired deviation products are `[2,0,2]`; sample covariance is $4/(3-1)=2$. Correlation divides covariance by the two sample standard deviations: $2/(1\times2)=1$.

Correlation describes linear association and is unitless; covariance retains units. Correlation is undefined when either variable is constant. A curved relationship can have low linear correlation, and correlation alone does not establish causation: another variable or the sampling process could explain it.

### 6. Percentiles and bootstrap intervals

A percentile describes a position in ordered data. The 50th percentile is the median. Libraries differ in how they interpolate between observed values; declare the convention. A percentile of observations and an uncertainty interval for the **mean** answer different questions.

A bootstrap repeatedly samples $n$ observations **with replacement** from the original $n$ observations, computes the statistic, and examines its distribution. In one resample of `[2,4,6]`, `[2,2,6]` is allowed: its mean is 10/3. With many resamples, a simple percentile interval takes the 2.5th and 97.5th percentiles of resampled means. This is an approximate method whose quality depends on sample size, sampling assumptions, and the statistic.

A frequentist 95% confidence procedure aims to cover a fixed population quantity in 95% of repeated datasets under its assumptions. It is not automatically a 95% probability statement about that fixed quantity given this particular interval. IID resampling also cannot repair a biased sample or account for time dependence; resample meaningful groups/blocks where appropriate.

### 7. Hypothesis and permutation tests

A **null hypothesis** describes the comparison model, such as no difference between exchangeable groups. Compute the observed absolute difference between group means. Shuffle group labels many times, keeping group sizes fixed, and recompute that difference. The fraction at least as extreme estimates how surprising the observation is under that null. The course uses $(extreme+1)/(repeats+1)$ to avoid reporting zero from finitely many random shuffles.

A p-value is not the probability the null hypothesis is true, and a small p-value does not measure practical importance. Report the effect size too. Running many tests and reporting only the smallest p-value creates false discoveries; specify comparisons in advance or use an appropriate correction. Paired or time-dependent data need a permutation scheme that respects their structure.

### Self-check

1. A coin succeeds with probability .2. What is failure probability, and the expected success count in 50 independent trials?
2. Why is sample variance of one number undefined?
3. If the original sample excludes an entire customer group, will 10,000 bootstrap resamples fix that?
4. Does a p-value of .03 mean the null has a 3% probability of being true?

<details><summary>Answers and reasoning</summary>

1. .8 and 10. An expectation is a long-run average, not a guaranteed observed count.
2. Its denominator $n-1$ is zero, and one observation cannot estimate variation around an unknown mean.
3. No. Resampling cannot invent the missing group's distribution.
4. No. It describes extremeness under the null and the stated sampling procedure, not a posterior probability for the null.

</details>


## Lesson

A random variable maps outcomes to numbers. Bernoulli models one binary trial; binomial counts successes across independent trials; a normal distribution is continuous and symmetric. Independence means observing one variable does not change the distribution of another. Conditional probability P(A|B) restricts attention to B. Bayes rule reverses conditioning using a base rate.

The mean is the arithmetic center; median resists extreme outliers; mode is the most frequent value. Variance measures squared deviations; standard deviation restores the original units. Sample variance uses n-1 when estimating population variance from an IID sample. Covariance tracks joint variation; correlation normalizes it and does not establish causation.

A confidence interval is a property of a repeated-sampling procedure, not the probability that a fixed parameter lies in this one observed interval. Bootstrap intervals resample observed data and inherit its biases. A p-value is the probability, under a specified null model, of a statistic at least as extreme as observed; it is not the probability the null is true.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import statistics
sample = [2, 4, 4, 6]
print(statistics.mean(sample), statistics.median(sample), statistics.variance(sample))


## Exercise 1: Sample variance

Compute unbiased sample variance for n >= 2; reject smaller samples.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def variance(xs):
    raise NotImplementedError


In [ ]:
assert variance([1, 2, 3]) == 1
expect_error(ValueError, lambda: variance([1]))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Compute the mean first.

</details>


## Exercise 2: Bayesian alert

Return P(event|positive) from prevalence, sensitivity, and false_positive_rate in [0,1]. Reject an impossible positive event.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def posterior(prevalence, sensitivity, false_positive_rate):
    raise NotImplementedError


In [ ]:
assert abs(posterior(.01, .9, .1)-(.009/.108)) < 1e-12
expect_error(ValueError, lambda: posterior(0, 1, 0))
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Count true positives and false positives in an imagined population.

</details>


## Exercise 3: Bootstrap mean interval

Return the 2.5th and 97.5th percentile order statistics of 1000 bootstrap means using seed=42. Require nonempty data.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def bootstrap_ci(xs, repeats=1000, seed=42):
    raise NotImplementedError


In [ ]:
lo, hi = bootstrap_ci([1, 2, 3, 4, 5])
assert lo <= 3 <= hi
assert bootstrap_ci([2]*10) == (2, 2)
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Resample with replacement, keeping sample size fixed.

</details>


## Independent transfer

Simulate 10,000 Bernoulli trials, plot a binomial count histogram, calculate percentiles/covariance/correlation, and run a permutation test of a mean difference. Explain sampling bias and multiple testing.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [02c interface and acceptance cases](../assignments/02c.md).
2. Copy the [blank starter](../assignments/starters/02c.py) to `work/assignments/02c.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 02c --solution work/assignments/02c.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/02c.md#02c-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 02c --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. What is a base-rate error?

2. Why is correlation insufficient for causation?


## Reading and review

- [Primary reference 1](https://d2l.ai/chapter_preliminaries/probability.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
